In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,roc_auc_score

In [3]:
data=load_breast_cancer()
x=data.data
y=(data.target==0).astype(int)

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=2,stratify=y)
print(x_train.shape)
print(y_train.shape)


(455, 30)
(455,)


In [7]:
def objective(trial):
    c=trial.suggest_float('c',1e-4,1e3,log=True)

    class_weight=trial.suggest_categorical('class_weight',['none','balanced'])
    if class_weight=='none':
      class_weight=None
    tol=trial.suggest_float('tol',1e-5,1e-2,log=True)

    model=make_pipeline(StandardScaler(),LogisticRegression(C=c,class_weight=class_weight,tol=tol,solver='lbfgs',max_iter=100,random_state=42))
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    scores=cross_val_score(model,x_train,y_train,cv=cv,scoring='roc_auc',n_jobs=-1)

    return scores.mean()

In [8]:
import numpy as np
import optuna
study=optuna.create_study(direction='maximize',study_name='logistic_regression_tuning')
study.optimize(objective,n_trials=50)
print('best roc-auc',study.best_value)
print('best hyperparameters:',study.best_params)

[I 2026-10-23 20:13:01,622] A new study created in memory with name: logistic_regression_tuning


[I 2026-10-23 20:13:01,945] Trial 0 finished with value: 0.996078431372549 and parameters: {'c': 0.0211980115976987, 'class_weight': 'balanced', 'tol': 0.007652471760986262}. Best is trial 0 with value: 0.996078431372549.
[I 2026-10-23 20:13:02,203] Trial 1 finished with value: 0.9894736842105264 and parameters: {'c': 0.0010810285779680887, 'class_weight': 'none', 'tol': 0.005949418362038118}. Best is trial 0 with value: 0.996078431372549.
[I 2026-10-23 20:13:02,262] Trial 2 finished with value: 0.9867905056759547 and parameters: {'c': 104.75425972585575, 'class_weight': 'none', 'tol': 0.0017483048917090876}. Best is trial 0 with value: 0.996078431372549.
[I 2026-10-23 20:13:02,332] Trial 3 finished with value: 0.9826625386996903 and parameters: {'c': 94.34022351135516, 'class_weight': 'none', 'tol': 0.00014822421842783335}. Best is trial 0 with value: 0.996078431372549.
[I 2026-10-23 20:13:02,386] Trial 4 finished with value: 0.9965944272445821 and parameters: {'c': 0.0795472632349156

best roc-auc 0.9968008255933952
best hyperparameters: {'c': 0.08783722673670011, 'class_weight': 'none', 'tol': 0.0020809848095430525}
